# PINN on CARLA — roundabout-augmented retrain + outage test

The synthetic-only model mistook the roundabout's long sustained turn for sensor bias and drifted **worse** than plain dead-reckoning (29 m vs 20 m). This version adds **roundabout-like motion** to the training set and trains on a much bigger dataset, then re-runs the exact same CARLA outage test.

**Run it overnight with `Save Version → Save & Run All (Commit)`** — that runs headless for up to 12 h. An interactive tab dies after ~40 min idle. A checkpoint is saved every `CKPT_EVERY` epochs to `MODEL_PATH`, so progress survives if you stop it.

In [ ]:
# ============================ CONFIG ============================
# Your two raw CARLA recordings (these paths already worked for you).
GROUND_TRUTH    = "/kaggle/input/datasets/aminemoussi1/carla-sensors/ground_truth.txt"   # true X,Y per tick
VEHICLE_SENSORS = "/kaggle/input/datasets/aminemoussi1/carla-sensors/vehicle_sensors.txt"

OUTAGE  = (4.25, 33.0)    # GNSS-outage window in seconds (PINN takes over from GNSS)
DT_PHYS = 0.00825         # CARLA fixed_delta -- the recording's tick rate (121 Hz)
DT      = 0.1             # PINN rate (10 Hz)
INTEGRATOR = "closed_form"   # your v7 winner

# --- BIG overnight training, WITH roundabout motion added ---
N_TRAIN, N_VAL, N_EPOCHS = 3000, 120, 800     # large run; finishes overnight with margin
P_ROUND     = 0.35                            # fraction of training windows that are roundabout-like
LAMBDA_BIAS = 100_000.0
RETRAIN     = True                            # train FRESH with the new mixed data
MODEL_PATH  = "/kaggle/working/pinn_round_big.pt"   # new file; your old model is untouched
CKPT_EVERY  = 50                              # checkpoint every N epochs (overnight safety)

N_EVAL_SEEDS = 50         # noise realizations averaged in the CARLA test (time is free now)
WILL_PEAK    = 2.0        # Will's FUSED-system peak error, drawn as a reference line
# ================================================================
print("config OK -- integrator:", INTEGRATOR, "| N_TRAIN:", N_TRAIN, "| epochs:", N_EPOCHS, "| p_round:", P_ROUND)

In [ ]:
# --- fail fast: make sure the data is reachable BEFORE training all night ---
import os
for name, p in [("ground_truth", GROUND_TRUTH), ("vehicle_sensors", VEHICLE_SENSORS)]:
    assert os.path.exists(p), "MISSING %s -> %s  (fix the path before running)" % (name, p)
    print("OK  %-16s %s" % (name, p))
print("data paths OK -- safe to start the long run")

In [ ]:
import numpy as np
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Building blocks (from your v7 notebook) + roundabout generator

In [ ]:
def _prim_straight(n_steps):
    return np.zeros(n_steps), np.zeros(n_steps)

def _prim_arc(n_steps, v_in, radius, direction):
    # Constant-radius turn at the entering speed; ax=0 keeps v constant.
    wz_val = direction * v_in / radius
    return np.zeros(n_steps), np.full(n_steps, wz_val)

def _prim_speed_change(n_steps, target_ax):
    return np.full(n_steps, target_ax), np.zeros(n_steps)


def compose_window(primitives_spec, v0, yaw0, dt=0.1, duration=30.0, integrator='euler'):
    """integrator: 'euler' or 'closed_form'. Both produce same IMU, different x_gt/y_gt."""
    N = int(round(duration / dt)) + 1
    ax_arr = np.zeros(N)
    wz_arr = np.zeros(N)

    n_steps_per = [int(round(p[2] / dt)) for p in primitives_spec]
    n_steps_per[-1] += (N - 1) - sum(n_steps_per)

    v_now = v0
    idx = 0
    for (ptype, kwargs, _), n_steps in zip(primitives_spec, n_steps_per):
        if ptype == 'straight':
            ax_seg, wz_seg = _prim_straight(n_steps)
        elif ptype == 'arc':
            ax_seg, wz_seg = _prim_arc(n_steps, v_now, kwargs['radius'], kwargs['direction'])
        elif ptype == 'speed_change':
            ax_seg, wz_seg = _prim_speed_change(n_steps, kwargs['ax'])
            v_now = v_now + kwargs['ax'] * (n_steps * dt)
        else:
            raise ValueError(ptype)
        ax_arr[idx:idx + n_steps] = ax_seg
        wz_arr[idx:idx + n_steps] = wz_seg
        idx += n_steps
    ax_arr[-1] = ax_arr[-2]; wz_arr[-1] = wz_arr[-2]   # avoid trailing-zero cosmetic

    # Integrate forward using the requested scheme
    x = np.zeros(N); y = np.zeros(N); v = np.zeros(N); yaw = np.zeros(N)
    x[0], y[0], v[0], yaw[0] = 0.0, 0.0, v0, yaw0
    eps = 1e-6
    for i in range(N - 1):
        a, w   = ax_arr[i], wz_arr[i]
        v_i, psi = v[i], yaw[i]
        if integrator == 'euler':
            x[i+1] = x[i] + v_i * np.cos(psi) * dt
            y[i+1] = y[i] + v_i * np.sin(psi) * dt
        elif integrator == 'closed_form':
            if abs(w) < eps:
                x[i+1] = x[i] + v_i * np.cos(psi) * dt
                y[i+1] = y[i] + v_i * np.sin(psi) * dt
            else:
                x[i+1] = x[i] + (v_i / w) * (np.sin(psi + w*dt) - np.sin(psi))
                y[i+1] = y[i] - (v_i / w) * (np.cos(psi + w*dt) - np.cos(psi))
        elif integrator == 'rk4':
            def _d(xx, yy, vv, pp):
                return vv*np.cos(pp), vv*np.sin(pp), a, w
            k1 = _d(x[i], y[i], v_i, psi)
            k2 = _d(x[i]+0.5*dt*k1[0], y[i]+0.5*dt*k1[1], v_i+0.5*dt*k1[2], psi+0.5*dt*k1[3])
            k3 = _d(x[i]+0.5*dt*k2[0], y[i]+0.5*dt*k2[1], v_i+0.5*dt*k2[2], psi+0.5*dt*k2[3])
            k4 = _d(x[i]+dt*k3[0],     y[i]+dt*k3[1],     v_i+dt*k3[2],     psi+dt*k3[3])
            x[i+1] = x[i] + dt/6.0*(k1[0]+2*k2[0]+2*k3[0]+k4[0])
            y[i+1] = y[i] + dt/6.0*(k1[1]+2*k2[1]+2*k3[1]+k4[1])
        else:
            raise ValueError(integrator)
        yaw[i+1] = psi + w * dt
        v[i+1]   = v_i + a * dt
    ay_arr = v * wz_arr
    t = np.arange(0, duration + dt/2, dt)

    return {
        't': t, 'dt': np.full(N, dt),
        'x_gt': x, 'y_gt': y, 'v_gt': v, 'yaw_gt': yaw,
        'ax': ax_arr, 'ay': ay_arr, 'wz': wz_arr,
        'v0': float(v0), 'yaw0': float(yaw0),
        'drive': 'synthetic_composed',
        'gt_integrator': integrator,
    }


def sample_complexity_window(complexity, seed, integrator='euler'):
    """Same as before but propagates integrator to compose_window."""
    rng = np.random.default_rng(seed)
    v0   = float(rng.uniform(5.0, 20.0))
    yaw0 = float(rng.uniform(-np.pi, np.pi))
    if complexity == 'simple':
        n_prims      = int(rng.choice([1, 2]))
        prim_probs   = {'straight': 0.60, 'arc': 0.20, 'speed_change': 0.20}
        radius_range = (60.0, 150.0); ax_range = (-0.3, 0.3)
    else:
        n_prims      = int(rng.choice([3, 4, 5]))
        prim_probs   = {'straight': 0.30, 'arc': 0.40, 'speed_change': 0.30}
        radius_range = (20.0, 100.0); ax_range = (-1.0, 1.0)
    names = list(prim_probs.keys())
    probs = np.array(list(prim_probs.values())); probs /= probs.sum()
    raw = rng.uniform(0.7, 1.3, n_prims); raw *= 30.0 / raw.sum()
    durations = raw.tolist()
    while min(durations) < 3.0 and len(durations) > 1:
        i = int(np.argmin(durations))
        j = i + 1 if i + 1 < len(durations) else i - 1
        durations[j] += durations[i]; durations.pop(i)
    s = sum(durations); durations = [d * 30.0 / s for d in durations]
    spec = []
    for d in durations:
        ptype = str(rng.choice(names, p=probs))
        if ptype == 'arc':
            kw = {'radius': float(rng.uniform(*radius_range)),
                  'direction': float(rng.choice([-1.0, 1.0]))}
        elif ptype == 'speed_change':
            kw = {'ax': float(rng.uniform(*ax_range))}
        else:
            kw = {}
        spec.append((ptype, kw, d))
    w = compose_window(spec, v0=v0, yaw0=yaw0, integrator=integrator)
    w['spec']       = spec
    w['complexity'] = complexity
    return w, spec


def make_dataset_v2(n_windows, seed_base=0, p_simple=0.8, integrator='euler'):
    rng = np.random.default_rng(seed_base)
    windows = []
    for i in range(n_windows):
        complexity = 'simple' if rng.random() < p_simple else 'complex'
        w, spec = sample_complexity_window(complexity, seed=seed_base + i, integrator=integrator)
        windows.append(w)
    n_simp = sum(1 for w in windows if w['complexity'] == 'simple')
    print(f"make_dataset_v2(n={n_windows}, integrator={integrator!r}):  "
          f"simple={n_simp}, complex={len(windows)-n_simp}")
    return windows

In [ ]:
# --- roundabout-like motion: long sustained turns, so the PINN sees the real driving regime ---
# A roundabout window = straight lead-in  ->  long tight sustained arc  ->  straight exit,
# which is exactly the motion shape that broke the synthetic-only model.
def sample_roundabout_window(seed, integrator="euler"):
    rng = np.random.default_rng(seed)
    v0     = float(rng.uniform(5.0, 11.0))           # roundabout speeds
    yaw0   = float(rng.uniform(-np.pi, np.pi))
    radius = float(rng.uniform(12.0, 30.0))          # tight radii (your loop was ~20 m)
    direction = float(rng.choice([-1.0, 1.0]))
    lead = float(rng.uniform(3.0, 7.0))              # straight approach
    arc  = float(rng.uniform(14.0, 22.0))            # long sustained turn
    out  = max(30.0 - lead - arc, 1.0)               # straight exit
    spec = [("straight", {}, lead),
            ("arc", {"radius": radius, "direction": direction}, arc),
            ("straight", {}, out)]
    w = compose_window(spec, v0=v0, yaw0=yaw0, integrator=integrator)
    w["complexity"] = "roundabout"; w["spec"] = spec
    return w, spec

def make_dataset_mixed(n, seed_base=0, p_round=0.30, p_simple=0.8, integrator="euler"):
    # p_round of windows are roundabout-like; the rest are the usual simple/complex mix.
    rng = np.random.default_rng(seed_base)
    W = []
    for i in range(n):
        if rng.random() < p_round:
            w, _ = sample_roundabout_window(seed_base + i, integrator=integrator)
        else:
            comp = "simple" if rng.random() < p_simple else "complex"
            w, _ = sample_complexity_window(comp, seed=seed_base + i, integrator=integrator)
        W.append(w)
    nr = sum(1 for w in W if w["complexity"] == "roundabout")
    print("make_dataset_mixed(n=%d): roundabout=%d, other=%d" % (n, nr, n - nr))
    return W

In [ ]:
NOISE_PARAMS_C = dict(
    sigma_b0_a=0.10,
    sigma_b0_w=0.015,
    sigma_a=0.05,          # white noise on accel (m/s²)
    sigma_w=0.01,          # white noise on gyro  (rad/s)
    sigma_brwa=0.001,      # accel bias random walk (m/s²/√s)
    sigma_brww=0.0002,     # gyro bias random walk  (rad/s/√s)
)

def inject_full_noise(window, noise_params, seed):
    """Constant bias + white noise + bias random walk."""
    rng = np.random.default_rng(seed)
    N = len(window['t'])
    dt = float(window['dt'][0])
    b0_ax = rng.normal(0.0, noise_params['sigma_b0_a'])
    b0_wz = rng.normal(0.0, noise_params['sigma_b0_w'])
    white_ax = rng.normal(0.0, noise_params['sigma_a'], N)
    white_wz = rng.normal(0.0, noise_params['sigma_w'], N)
    brw_ax = np.cumsum(rng.normal(0.0, noise_params['sigma_brwa']*np.sqrt(dt), N))
    brw_wz = np.cumsum(rng.normal(0.0, noise_params['sigma_brww']*np.sqrt(dt), N))
    bias_a_profile = b0_ax + brw_ax
    bias_w_profile = b0_wz + brw_wz
    return {
        'ax': window['ax'] + bias_a_profile + white_ax,
        'ay': window['ay'],
        'wz': window['wz'] + bias_w_profile + white_wz,
        'bias_a_profile': bias_a_profile,
        'bias_w_profile': bias_w_profile,
    }


def stack_batch_full_noise_v2(windows, noise_params, base_seed):
    """Same as stack_batch_full_noise but also returns per-timestep bias profiles."""
    noisy = [inject_full_noise(w, noise_params, base_seed + i) for i, w in enumerate(windows)]
    ax = torch.tensor(np.stack([n['ax'] for n in noisy]), dtype=torch.float32, device=DEVICE)
    ay = torch.tensor(np.stack([n['ay'] for n in noisy]), dtype=torch.float32, device=DEVICE)
    wz = torch.tensor(np.stack([n['wz'] for n in noisy]), dtype=torch.float32, device=DEVICE)
    dt = torch.tensor(np.stack([w['dt'] for w in windows]), dtype=torch.float32, device=DEVICE)
    v0 = torch.tensor([w['v0']   for w in windows], dtype=torch.float32, device=DEVICE)
    yaw0 = torch.tensor([w['yaw0'] for w in windows], dtype=torch.float32, device=DEVICE)
    b_a_mean = torch.tensor([n['bias_a_profile'].mean() for n in noisy],
                            dtype=torch.float32, device=DEVICE)
    b_w_mean = torch.tensor([n['bias_w_profile'].mean() for n in noisy],
                            dtype=torch.float32, device=DEVICE)
    # NEW: per-timestep bias profile tensors (B, T)
    ba_profile = torch.tensor(np.stack([n['bias_a_profile'] for n in noisy]),
                              dtype=torch.float32, device=DEVICE)
    bw_profile = torch.tensor(np.stack([n['bias_w_profile'] for n in noisy]),
                              dtype=torch.float32, device=DEVICE)
    return ax, ay, wz, dt, v0, yaw0, b_a_mean, b_w_mean, ba_profile, bw_profile

In [ ]:
class BiasGRU(nn.Module):
    def __init__(self, hidden=32, n_layers=1):
        super().__init__()
        self.gru  = nn.GRU(input_size=6, hidden_size=hidden,
                           num_layers=n_layers, batch_first=True)
        self.head = nn.Linear(hidden, 2)
        # Small (NOT zero) weights so initial bias predictions are tiny
        # but gradients can flow back into the GRU.
        nn.init.normal_(self.head.weight, std=1e-3)
        nn.init.zeros_(self.head.bias)

    def forward(self, imu_seq, state_seq):
        x = torch.cat([imu_seq, state_seq], dim=-1)
        h, _ = self.gru(x)
        bias = self.head(h)
        return bias


class GreyBoxPINN(nn.Module):
    def __init__(self, hidden=32, integrator='euler'):
        super().__init__()
        self.bias_net = BiasGRU(hidden=hidden)
        self.integrator = integrator

    def forward(self, ax, ay, wz, dt, v0, yaw0):
        B, T = ax.shape
        v0_seq     = v0.unsqueeze(1).expand(B, T)
        sin_yaw0_s = torch.sin(yaw0).unsqueeze(1).expand(B, T)
        cos_yaw0_s = torch.cos(yaw0).unsqueeze(1).expand(B, T)
        imu_seq    = torch.stack([ax, ay, wz], dim=-1)
        state_seq  = torch.stack([v0_seq, sin_yaw0_s, cos_yaw0_s], dim=-1)
        bias_seq   = self.bias_net(imu_seq, state_seq)

        x_list   = [torch.zeros(B, device=ax.device)]
        y_list   = [torch.zeros(B, device=ax.device)]
        v_list   = [v0]
        yaw_list = [yaw0]
        eps = 1e-4

        for i in range(T - 1):
            b_a = bias_seq[:, i, 0]; b_w = bias_seq[:, i, 1]
            ax_clean = ax[:, i] - b_a
            wz_clean = wz[:, i] - b_w
            dt_i  = dt[:, i]
            v_cur = v_list[-1]; psi = yaw_list[-1]

            if self.integrator == 'euler':
                x_next = x_list[-1] + v_cur * torch.cos(psi) * dt_i
                y_next = y_list[-1] + v_cur * torch.sin(psi) * dt_i
            elif self.integrator == 'closed_form':
                psi_new = psi + wz_clean * dt_i
                mask    = torch.abs(wz_clean) < eps
                safe_w  = torch.where(mask, torch.full_like(wz_clean, eps), wz_clean)
                x_cf = x_list[-1] + v_cur * (torch.sin(psi_new) - torch.sin(psi)) / safe_w
                y_cf = y_list[-1] - v_cur * (torch.cos(psi_new) - torch.cos(psi)) / safe_w
                x_st = x_list[-1] + v_cur * torch.cos(psi) * dt_i
                y_st = y_list[-1] + v_cur * torch.sin(psi) * dt_i
                x_next = torch.where(mask, x_st, x_cf)
                y_next = torch.where(mask, y_st, y_cf)
            elif self.integrator == 'rk4':
                a = ax_clean; w = wz_clean
                def _d(xx, yy, vv, pp):
                    return vv*torch.cos(pp), vv*torch.sin(pp), a, w
                k1 = _d(x_list[-1], y_list[-1], v_cur, psi)
                k2 = _d(x_list[-1]+0.5*dt_i*k1[0], y_list[-1]+0.5*dt_i*k1[1], v_cur+0.5*dt_i*k1[2], psi+0.5*dt_i*k1[3])
                k3 = _d(x_list[-1]+0.5*dt_i*k2[0], y_list[-1]+0.5*dt_i*k2[1], v_cur+0.5*dt_i*k2[2], psi+0.5*dt_i*k2[3])
                k4 = _d(x_list[-1]+dt_i*k3[0],     y_list[-1]+dt_i*k3[1],     v_cur+dt_i*k3[2],     psi+dt_i*k3[3])
                x_next = x_list[-1] + dt_i/6.0*(k1[0]+2*k2[0]+2*k3[0]+k4[0])
                y_next = y_list[-1] + dt_i/6.0*(k1[1]+2*k2[1]+2*k3[1]+k4[1])
            else:
                raise ValueError(self.integrator)

            v_next   = v_cur + ax_clean * dt_i
            yaw_next = psi   + wz_clean * dt_i
            x_list.append(x_next); y_list.append(y_next)
            v_list.append(v_next); yaw_list.append(yaw_next)

        x   = torch.stack(x_list,   dim=1)
        y   = torch.stack(y_list,   dim=1)
        v   = torch.stack(v_list,   dim=1)
        yaw = torch.stack(yaw_list, dim=1)
        return x, y, v, yaw, bias_seq

In [ ]:
def train_noisy_with_bias(model, train_windows, val_windows, noise_params,
                          stack_fn=None,
                          n_epochs=600, batch_size=32, lr=1e-3, log_every=20,
                          lambda_smooth=100.0, lambda_bias=0.0,
                          ckpt_path=None, ckpt_every=50):
    """
    Trainer with optional direct bias supervision.
    lambda_bias = 0 -> same behaviour as train_noisy.
    lambda_bias > 0 -> add MSE(predicted_bias_profile, true_bias_profile) to the loss.
    stack_fn must return 10 values (see stack_batch_full_noise_v2).
    """
    if stack_fn is None:
        stack_fn = stack_batch_full_noise_v2

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    history = {'epoch': [], 'pos_loss': [], 'smooth_loss': [], 'bias_loss': [],
               'val_drift_30s': [], 'bias_corr_a': [], 'bias_corr_w': []}

    for ep in range(n_epochs):
        model.train()
        idxs = np.random.permutation(len(train_windows))
        epoch_seed = 1_000_000 + ep * 10_000
        last_pos, last_smooth, last_bias = 0.0, 0.0, 0.0

        for start in range(0, len(train_windows), batch_size):
            batch_idx = idxs[start:start + batch_size]
            batch = [train_windows[i] for i in batch_idx]
            ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b, _, _, ba_prof, bw_prof = stack_fn(
                batch, noise_params, epoch_seed + start)
            x_gt = torch.tensor(np.stack([w['x_gt'] for w in batch]),
                                dtype=torch.float32, device=DEVICE)
            y_gt = torch.tensor(np.stack([w['y_gt'] for w in batch]),
                                dtype=torch.float32, device=DEVICE)

            x_p, y_p, _, _, bias_seq = model(ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b)

            pos_loss    = ((x_p - x_gt)**2 + (y_p - y_gt)**2).mean()
            smooth_loss = ((bias_seq[:, 1:, :] - bias_seq[:, :-1, :])**2).mean()
            bias_loss   = ((bias_seq[:, :, 0] - ba_prof)**2 +
                           (bias_seq[:, :, 1] - bw_prof)**2).mean()

            loss = pos_loss + lambda_smooth * smooth_loss + lambda_bias * bias_loss

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            last_pos    = pos_loss.item()
            last_smooth = smooth_loss.item()
            last_bias   = bias_loss.item()
        scheduler.step()
        if ckpt_path is not None and (ep % ckpt_every == 0 or ep == n_epochs - 1):
            torch.save(model.state_dict(), ckpt_path)

        if ep % log_every == 0 or ep == n_epochs - 1:
            model.eval()
            with torch.no_grad():
                val_drifts, true_bas, pred_bas, true_bws, pred_bws = [], [], [], [], []
                for vw in val_windows:
                    for s in range(3):
                        ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v, b_a_t, b_w_t, _, _ = stack_fn(
                            [vw], noise_params, base_seed=900_000 + s * 100)
                        x_v, y_v, _, _, bias_v = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
                        drift = float(torch.sqrt(
                            (x_v[0, -1] - vw['x_gt'][-1])**2 +
                            (y_v[0, -1] - vw['y_gt'][-1])**2))
                        val_drifts.append(drift)
                        true_bas.append(float(b_a_t[0]))
                        pred_bas.append(float(bias_v[:,:,0].mean()))
                        true_bws.append(float(b_w_t[0]))
                        pred_bws.append(float(bias_v[:,:,1].mean()))
                corr_a = np.corrcoef(true_bas, pred_bas)[0, 1]
                corr_w = np.corrcoef(true_bws, pred_bws)[0, 1]

            history['epoch'].append(ep)
            history['pos_loss'].append(last_pos)
            history['smooth_loss'].append(last_smooth)
            history['bias_loss'].append(last_bias)
            history['val_drift_30s'].append(float(np.mean(val_drifts)))
            history['bias_corr_a'].append(float(corr_a))
            history['bias_corr_w'].append(float(corr_w))
            print(f"  ep {ep:4d}  pos={last_pos:7.2f}  bias={last_bias:.5f}  "
                  f"val_drift={np.mean(val_drifts):6.2f}m  "
                  f"corr(b_a)={corr_a:+.3f}  corr(b_w)={corr_w:+.3f}  "
                  f"lr={opt.param_groups[0]['lr']:.5f}")
    return history

## 1. Train the PINN on synthetic + roundabout motion

`RETRAIN=True` trains fresh on the mixed data. A checkpoint lands in `MODEL_PATH` every `CKPT_EVERY` epochs.

In [ ]:
import os, torch
model = GreyBoxPINN(integrator=INTEGRATOR).to(DEVICE)
history = None
if (not RETRAIN) and os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE)); print("loaded", MODEL_PATH)
else:
    print("training (%s) on synthetic + roundabout motion: N_TRAIN=%d, epochs=%d, p_round=%.2f"
          % (INTEGRATOR, N_TRAIN, N_EPOCHS, P_ROUND))
    train_w = make_dataset_mixed(N_TRAIN, seed_base=0,       p_round=P_ROUND, integrator=INTEGRATOR)
    val_w   = make_dataset_mixed(N_VAL,   seed_base=500_000, p_round=P_ROUND, integrator=INTEGRATOR)
    history = train_noisy_with_bias(model, train_w, val_w, NOISE_PARAMS_C,
                                    n_epochs=N_EPOCHS, lambda_bias=LAMBDA_BIAS,
                                    log_every=50, ckpt_path=MODEL_PATH, ckpt_every=CKPT_EVERY)
    torch.save(model.state_dict(), MODEL_PATH); print("saved final model:", MODEL_PATH)
model.eval(); print("model ready")

## 2. Build the CARLA outage window

In [ ]:
import numpy as np

def parse_pairs(path, n):
    rows = []
    for line in open(path):
        p = [s.strip() for s in line.strip().split(",")]
        if len(p) < 2 * n:
            continue
        try:
            rows.append([float(p[2 * i + 1]) for i in range(n)])
        except ValueError:
            continue
    return np.array(rows)

def smooth(a, w=5):
    pad = w // 2
    return np.convolve(np.pad(a, pad, mode="edge"), np.ones(w) / w, mode="valid")

# derive clean IMU + speed/yaw + true path from the raw recording, all on a 10 Hz grid
veh = parse_pairs(VEHICLE_SENSORS, 3)            # steer, speed, yaw(deg)
gt  = parse_pairs(GROUND_TRUTH, 3)               # x, y, z
t_p  = np.arange(len(veh)) * DT_PHYS
t_gt = np.arange(len(gt)) * DT_PHYS

t   = np.arange(t_p[0], t_p[-1], DT)
spd = smooth(np.interp(t, t_p, veh[:, 1]))
yaw = smooth(np.interp(t, t_p, np.unwrap(np.deg2rad(veh[:, 2]))))
ax_c = np.gradient(spd, DT)
wz_c = np.gradient(yaw, DT)
ay_c = spd * wz_c
gx = np.interp(t, t_gt, gt[:, 0]); gy = np.interp(t, t_gt, gt[:, 1])

i0, i1 = int(np.searchsorted(t, OUTAGE[0])), int(np.searchsorted(t, OUTAGE[1]))
sl = slice(i0, i1 + 1)
t_o = t[sl]; N = len(t_o)
v0, yaw0 = float(spd[i0]), float(yaw[i0])
xg = gx[sl] - gx[i0]; yg = gy[sl] - gy[i0]       # true path relative to the dropout point

carla = dict(t=t_o, dt=np.full(N, DT), ax=ax_c[sl], ay=ay_c[sl], wz=wz_c[sl],
             v0=v0, yaw0=yaw0, x_gt=xg, y_gt=yg)
print("outage window: %d samples (%.1f s), v0=%.1f m/s, yaw0=%.0f deg"
      % (N, t_o[-1] - t_o[0], v0, np.degrees(yaw0)))


## 3. Run the PINN + baselines

In [ ]:
import numpy as np, torch

def dr_closed_form(ax, wz, v0, yaw0, dt):
    # numpy closed-form dead-reckon from (0,0,v0,yaw0) -- same math as the PINN integrator
    N = len(ax); x = np.zeros(N); y = np.zeros(N); v = v0; psi = yaw0; eps = 1e-6
    for i in range(N - 1):
        w = wz[i]
        if abs(w) < eps:
            x[i + 1] = x[i] + v * np.cos(psi) * dt; y[i + 1] = y[i] + v * np.sin(psi) * dt
        else:
            x[i + 1] = x[i] + (v / w) * (np.sin(psi + w * dt) - np.sin(psi))
            y[i + 1] = y[i] - (v / w) * (np.cos(psi + w * dt) - np.cos(psi))
        psi += w * dt; v += ax[i] * dt
    return x, y

def run_pinn(ax, ay, wz, v0, yaw0, dt):
    A = lambda a: torch.tensor(a[None, :], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        xp, yp, _, _, bias = model(A(ax), A(ay), A(wz), A(np.full(len(ax), dt)),
                                   torch.tensor([v0], dtype=torch.float32, device=DEVICE),
                                   torch.tensor([yaw0], dtype=torch.float32, device=DEVICE))
    return xp[0].cpu().numpy(), yp[0].cpu().numpy(), bias[0].cpu().numpy()

xg, yg = carla["x_gt"], carla["y_gt"]
xf, yf = dr_closed_form(carla["ax"], carla["wz"], v0, yaw0, DT)   # clean-IMU floor (no noise)
floor_drift = np.hypot(xf - xg, yf - yg)

pinn_peaks, naive_peaks, rep = [], [], None
for s in range(N_EVAL_SEEDS):
    noisy = inject_full_noise(carla, NOISE_PARAMS_C, seed=12345 + s)
    xn, yn = dr_closed_form(noisy["ax"], noisy["wz"], v0, yaw0, DT)               # naive: noisy, no bias removal
    xp, yp, bias = run_pinn(noisy["ax"], noisy["ay"], noisy["wz"], v0, yaw0, DT)  # PINN: bias removed
    naive_peaks.append(float(np.hypot(xn - xg, yn - yg).max()))
    pinn_peaks.append(float(np.hypot(xp - xg, yp - yg).max()))
    if s == 0:
        rep = dict(xn=xn, yn=yn, xp=xp, yp=yp, wzn=noisy["wz"], bias=bias, bw=noisy["bias_w_profile"])
pinn_peaks = np.array(pinn_peaks); naive_peaks = np.array(naive_peaks)

print("=== peak drift over the outage (%d noise realizations) ===" % N_EVAL_SEEDS)
print("clean-IMU floor : %5.2f m   (best case, perfect IMU)" % floor_drift.max())
print("naive (no PINN) : %5.2f +/- %.2f m   (worst %.2f)" % (naive_peaks.mean(), naive_peaks.std(), naive_peaks.max()))
print("PINN            : %5.2f +/- %.2f m   (worst %.2f)" % (pinn_peaks.mean(), pinn_peaks.std(), pinn_peaks.max()))
print("Will (fused ref): %5.2f m" % WILL_PEAK)


## 4. Plots

In [ ]:
import matplotlib.pyplot as plt, numpy as np
r = rep; tt = t_o - t_o[0]
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

a = axs[0, 0]
a.plot(xg, yg, "k-", lw=2.5, label="true (GPS)")
a.plot(r["xp"], r["yp"], "g-", lw=1.8, label="PINN (peak %.2f m)" % pinn_peaks.mean())
a.plot(r["xn"], r["yn"], "r--", lw=1.4, label="naive dead-reckon (peak %.2f m)" % naive_peaks.mean())
a.scatter([0], [0], c="b", s=60, zorder=5, label="GNSS drops")
a.set_aspect("equal"); a.legend(); a.set_xlabel("x (m)"); a.set_ylabel("y (m)")
a.set_title("Trajectory during outage (start-relative)")

a = axs[0, 1]
a.plot(tt, np.hypot(r["xp"] - xg, r["yp"] - yg), "g-", label="PINN")
a.plot(tt, np.hypot(r["xn"] - xg, r["yn"] - yg), "r--", label="naive dead-reckon")
a.plot(tt, floor_drift, "-", color="grey", alpha=.7, label="clean-IMU floor")
a.axhline(WILL_PEAK, color="purple", ls=":", label="Will (fused) %.0f m" % WILL_PEAK)
a.legend(); a.set_xlabel("time into outage (s)"); a.set_ylabel("position error (m)")
a.set_title("Drift growth over the outage")

a = axs[1, 0]
a.plot(tt, carla["wz"], "k-", label="clean wz")
a.plot(tt, r["wzn"], "r-", alpha=.5, label="noisy wz (PINN input)")
a.legend(); a.set_xlabel("time into outage (s)"); a.set_ylabel("wz (rad/s)")
a.set_title("Gyro: clean vs noisy (what the PINN sees)")

a = axs[1, 1]
a.plot(tt, r["bw"], "k-", label="injected gyro bias")
a.plot(tt, r["bias"][:, 1], "g-", label="PINN-estimated bias")
a.legend(); a.set_xlabel("time into outage (s)"); a.set_ylabel("gyro bias (rad/s)")
a.set_title("Bias the PINN removed")

plt.tight_layout(); plt.savefig("/kaggle/working/pinn_carla_result.png", dpi=110); plt.show()
print("saved -> /kaggle/working/pinn_carla_result.png")

if history is not None:
    plt.figure(figsize=(7, 4))
    plt.plot(history["epoch"], history["val_drift_30s"], "-o", ms=3)
    plt.xlabel("epoch"); plt.ylabel("val drift (m)"); plt.title("Training convergence (synthetic)")
    plt.tight_layout(); plt.show()


## How to read the plots

- **Top-left (trajectory):** true GPS path vs PINN vs naive dead-reckoning over the outage. Closer to black is better.
- **Top-right (drift vs time):** position error growing through the outage. The grey **clean-IMU floor** (~6 m) is the best *any* pure-IMU method can do over this 29 s. Will's 2 m (dotted) comes from *fusion* with vision — a later phase, not the target for this standalone test.
- **Bottom-left (gyro):** clean vs noisy yaw-rate — what the PINN actually sees.
- **Bottom-right (bias) — the key plot:** black = the real injected bias (steady); green = what the PINN estimated. **Success = green goes roughly flat near the black line** (tracking real bias, not the motion). If green still swings up/down with the turn, raise `P_ROUND` toward 0.45–0.5 and retrain.

**Goal:** PINN peak drift drops from 29 m toward the ~6 m floor, and clearly below the naive line.